# Modelos GARCH Assimetricos: EGARCH e GJR-GARCH

Neste notebook, exploramos modelos que capturam o **efeito alavancagem** (leverage effect):
a observacao empirica de que choques negativos (quedas de preco) tendem a aumentar mais
a volatilidade do que choques positivos de mesma magnitude.

**Modelos cobertos:**
- EGARCH de Nelson (1991)
- GJR-GARCH de Glosten, Jagannathan e Runkle (1993)

**Conteudo:**
1. Motivacao: efeito alavancagem
2. EGARCH de Nelson (1991)
3. Interpretando o parametro de assimetria
4. GJR-GARCH de Glosten, Jagannathan e Runkle (1993)
5. News Impact Curve
6. Comparacao GARCH vs EGARCH vs GJR
7. Aplicacao: dados do Ibovespa

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Motivacao: efeito alavancagem

O **efeito alavancagem** (leverage effect), documentado por Black (1976), refere-se a
correlacao negativa entre retornos e mudancas na volatilidade:

- Quando precos **caem**, a razao divida/patrimonio da empresa **aumenta** (maior alavancagem)
- Isso torna a empresa mais **arriscada**, elevando a volatilidade
- Choques **negativos** geram mais volatilidade que choques **positivos** de mesma magnitude

O GARCH(1,1) simetrico **nao captura** esse efeito — ele trata choques positivos e negativos
igualmente ($\epsilon_{t-1}^2$ ignora o sinal de $\epsilon_{t-1}$).

Vamos visualizar esse efeito nos dados.

In [ ]:
# Carregar dados do S&P 500
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']

# Demonstrar correlacao negativa retorno-volatilidade
# Volatilidade proxy: retornos ao quadrado em janela movel
rolling_vol = returns.rolling(21).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(returns.index, returns.values, linewidth=0.5)
axes[0].set_title('Retornos S&P 500')
axes[0].set_ylabel('Retorno')

axes[1].plot(rolling_vol.index, rolling_vol.values, color='darkorange', linewidth=0.8)
axes[1].set_title('Volatilidade movel (21 dias)')
axes[1].set_ylabel('Desvio Padrao')

fig.tight_layout()
plt.show()

# Correlacao retornos vs mudanca na volatilidade
vol_change = rolling_vol.diff()
corr = returns.corr(vol_change)
print(f"Correlacao entre retornos e mudanca na volatilidade: {corr:.4f}")
print("(Valor negativo confirma o efeito alavancagem)")

## 2. EGARCH de Nelson (1991)

O modelo **EGARCH** (Exponential GARCH) modela o **logaritmo** da variancia condicional:

$$\ln(\sigma_t^2) = \omega + \alpha \left( |z_{t-1}| - E|z_{t-1}| \right) + \gamma z_{t-1} + \beta \ln(\sigma_{t-1}^2)$$

onde $z_t = \epsilon_t / \sigma_t$ sao os residuos padronizados.

**Vantagens do EGARCH:**
- Nao requer restricoes de nao-negatividade nos parametros (modela $\ln \sigma^2$)
- O parametro $\gamma$ captura a assimetria: se $\gamma < 0$, choques negativos aumentam mais a volatilidade
- Naturalmente garante $\sigma_t^2 > 0$

In [ ]:
# TODO: Estime um EGARCH(1,1) com archbox
# Dicas:
# - model_egarch = EGARCH(returns.values, p=1, q=1)
# - results_egarch = model_egarch.fit()
# - print(results_egarch.summary())

## 3. Interpretando o parametro de assimetria $\gamma$

No EGARCH, o parametro $\gamma$ (gamma) mede a assimetria da resposta a choques:

- Se $\gamma = 0$: resposta simetrica (equivalente ao GARCH)
- Se $\gamma < 0$: choques negativos ($z_{t-1} < 0$) aumentam mais $\sigma_t^2$ → **efeito alavancagem**
- Se $\gamma > 0$: choques positivos aumentam mais $\sigma_t^2$ (raro em acoes)

Para um choque negativo ($z_{t-1} = -|z|$):
$$\text{impacto} = \alpha |z| - \gamma |z| = (\alpha - \gamma)|z|$$

Para um choque positivo ($z_{t-1} = |z|$):
$$\text{impacto} = \alpha |z| + \gamma |z| = (\alpha + \gamma)|z|$$

In [ ]:
# TODO: Extraia gamma e interprete o efeito alavancagem
# Dicas:
# - Acesse os parametros: results_egarch.params
# - Nomes: results_egarch.param_names
# - Verifique se gamma e significativo: results_egarch.pvalues
# - Se gamma < 0 e significativo, confirma efeito alavancagem

## 4. GJR-GARCH de Glosten, Jagannathan e Runkle (1993)

O modelo **GJR-GARCH** adiciona um termo de **threshold** ao GARCH padrao:

$$\sigma_t^2 = \omega + (\alpha + \gamma I_{t-1}) \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

onde $I_{t-1}$ e uma funcao indicadora:
$$I_{t-1} = \begin{cases} 1 & \text{se } \epsilon_{t-1} < 0 \\ 0 & \text{caso contrario} \end{cases}$$

**Interpretacao:**
- Choque positivo: impacto = $\alpha \epsilon^2$
- Choque negativo: impacto = $(\alpha + \gamma) \epsilon^2$
- Se $\gamma > 0$: choques negativos tem impacto maior → **efeito alavancagem**

**Condicao de estacionariedade:** $\alpha + \beta + \gamma/2 < 1$

In [ ]:
# TODO: Estime um GJR-GARCH(1,1) com archbox
# Dicas:
# - model_gjr = GJRGARCH(returns.values, p=1, q=1)
# - results_gjr = model_gjr.fit()
# - print(results_gjr.summary())
# - Verifique se gamma > 0 (efeito alavancagem no GJR)

## 5. News Impact Curve

A **News Impact Curve** (NIC) mostra como choques passados ($\epsilon_{t-1}$) afetam a
variancia condicional corrente ($\sigma_t^2$), mantendo toda informacao anterior constante.

Para o GARCH simetrico, a NIC e uma **parabola centrada em zero** — choques positivos e
negativos de mesma magnitude geram o mesmo impacto na volatilidade.

Para modelos assimetricos (EGARCH, GJR), a NIC e **assimetrica** — choques negativos
geram maior impacto. Comparar as NICs dos diferentes modelos e uma forma visual
poderosa de entender as diferencas entre eles.

In [ ]:
# TODO: Plote a news impact curve para GARCH, EGARCH e GJR
# Dicas:
# - Primeiro estime o GARCH(1,1) padrao como referencia
# - Use plot_news_impact(results_garch, title='NIC - GARCH(1,1)')
# - Sobreponha as curvas dos tres modelos em um unico grafico
# - Observe a assimetria nas curvas EGARCH e GJR

## 6. Comparacao GARCH vs EGARCH vs GJR

Podemos comparar os modelos usando **criterios de informacao**:

| Criterio | Formula | Penalidade |
|----------|---------|------------|
| AIC | $-2\ell + 2k$ | Leve |
| BIC | $-2\ell + k \ln(n)$ | Moderada |
| HQIC | $-2\ell + 2k \ln(\ln(n))$ | Intermediaria |

onde $\ell$ e a log-verossimilhanca, $k$ o numero de parametros e $n$ o numero de observacoes.

**Menor valor = melhor modelo.** O BIC penaliza mais modelos complexos que o AIC.

In [ ]:
# TODO: Compare AIC/BIC dos tres modelos
# Dicas:
# - Crie um DataFrame com os criterios de cada modelo:
#   comparison = pd.DataFrame({
#       'GARCH': [results_garch.aic, results_garch.bic, results_garch.hqic],
#       'EGARCH': [results_egarch.aic, results_egarch.bic, results_egarch.hqic],
#       'GJR': [results_gjr.aic, results_gjr.bic, results_gjr.hqic],
#   }, index=['AIC', 'BIC', 'HQIC'])
# - Use plot_model_comparison() para visualizar
# - Identifique o melhor modelo por cada criterio

## 7. Aplicacao: dados do Ibovespa

Agora aplique os tres modelos nos dados do **Ibovespa** (indice da bolsa brasileira).

Mercados emergentes como o Brasil tendem a ter:
- Maior volatilidade base
- Efeito alavancagem potencialmente mais forte
- Caudas mais pesadas

Compare os resultados com os do S&P 500.

In [ ]:
# TODO: Estime os tres modelos nos dados ibovespa_returns.csv
# Dicas:
# - ibov = pd.read_csv('../data/ibovespa_returns.csv', parse_dates=['date'], index_col='date')
# - ibov_returns = ibov['returns']
# - Estime GARCH, EGARCH e GJR
# - Compare os parametros com os do S&P 500
# - O efeito alavancagem e mais forte no Ibovespa?

## Conclusao

Neste notebook, aprendemos:

- O **efeito alavancagem** e por que modelos simetricos sao insuficientes
- O modelo **EGARCH**: modela $\ln(\sigma^2_t)$, parametro $\gamma$ captura assimetria
- O modelo **GJR-GARCH**: usa indicadora $I_{t-1}$ para diferenciar choques negativos
- A **News Impact Curve** como ferramenta visual para comparar modelos
- Como usar **criterios de informacao** para selecao de modelos

No proximo notebook, exploraremos o **APARCH** (potencia variavel) e o
**Component-GARCH** (decomposicao em componentes de curto e longo prazo).